Hugo Oropesiano Valadés - 100522384

Iker Mirón Blázquez - 100522189

# Práctica 1 — Predicción de Suscripción a un Producto Bancario


El objetivo de esta práctica es construir un modelo predictivo que determine si un cliente bancario suscribirá o no un depósito a partir de datos demográficos, financieros y de interacción. Se trabajará con el fichero `bank_17.pkl` (100522189; 8+9 = 17) para todo el proceso de entrenamiento, optimización y evaluación, y posteriormente se evaluará el modelo final con `bank_competition.pkl`.

A lo largo de la práctica se realizará un análisis de datos (EDA), un preprocesado, una selección de scaler, HPO con KNN, árboles de decisión, modelos lineales y SVM, una evaluación del modelo final y un despliegue con Streamlit.

Antes de comenzar, descargamos todas las dependencias necesarias

In [ ]:
pip install -r requirements.txt

## Fase 1 — EDA Simplificado

A continuación se realizará el análisis de los datos de entrenamiento. Antes de comenzar, convertimos el fichero `bank_17.pkl` a formato CSV para poder inspeccionar los datos fácilmente fuera del notebook (por ejemplo, en Excel).

In [ ]:
import pickle
import pandas as pd

# Cargar datos de entrenamiento
with open('Entrenamiento/bank_17.pkl', 'rb') as f:
    df = pickle.load(f)

# Cambiar a CSV
df.to_csv('Entrenamiento/bank_17.csv', index=False)
print("Archivo CSV correcto:Entrenamiento/bank_17.csv")

### 1.1 — Inspección y análisis del dataset

A continuación se comprobará con código el número de instancias y variables, el tipo de cada variable, cuáles son las variables ordinales y cuáles las categóricas con alta cardinalidad. Por último, se hará una búsqueda de variables con valores faltantes (NaN) y variables constantes.

In [ ]:
import pandas as pd
from IPython.display import display

# Nº de instancias y variables
n_instancias, n_variables = df.shape
resumen_dimensiones = pd.DataFrame({
    'Instancias': [n_instancias],
    'Variables totales': [n_variables],
    'Features': [n_variables - 1],
    'Variable objetivo': ['deposit']
})

# Tipo de cada variable
numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categoricas = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Variables ordinales conocidas del dominio
ordinales = ['education']  # primary < secondary < tertiary
categoricas_puras = [c for c in categoricas if c not in ordinales]

tabla_variables = pd.DataFrame({
    'Numéricas': pd.Series(numericas),
    'Categóricas': pd.Series(categoricas_puras),
    'Ordinales': pd.Series(ordinales)
})

# Variables categóricas con alta cardinalidad (>10 valores)
card = df[categoricas].nunique().sort_values(ascending=False)
card_df = pd.DataFrame({
    'variable': card.index,
    'valores_unicos': card.values,
    'alta_cardinalidad': (card.values > 10)
})

# Variables con valores faltantes (NaN o None)
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'variable': nulos.index,
    'nulos': nulos.values,
    'porcentaje (%)': nulos_pct.values
})
missing_df = missing_df[missing_df['nulos'] > 0]
if missing_df.empty:
    missing_df = pd.DataFrame([{
        'variable': 'Sin valores faltantes',
        'nulos': 0,
        'porcentaje (%)': 0.0
    }])

# Buscar NaNs o Nones en variables categóricas
nulos_obj = df[categoricas].isna().sum()
nulos_obj = nulos_obj[nulos_obj > 0]
nulos_obj_df = pd.DataFrame({
    'variable_categorica': nulos_obj.index,
    'nulos': nulos_obj.values
})
if nulos_obj_df.empty:
    nulos_obj_df = pd.DataFrame([{
        'variable_categorica': 'Sin nulos en categóricas',
        'nulos': 0
    }])

# Variables constantes o con valores todos únicos
constantes = [col for col in df.columns if df[col].nunique() <= 1]
id_cols = [col for col in df.columns if df[col].nunique() == len(df)]

constantes_ids_df = pd.DataFrame({
    'Constantes': pd.Series(constantes if constantes else ['Ninguna']),
    'Todos únicos (tipo ID)': pd.Series(id_cols if id_cols else ['Ninguna'])
})

# Tipo de problema y desbalanceo de clases
class_counts = df['deposit'].value_counts()
class_pct = (df['deposit'].value_counts(normalize=True) * 100).round(2)
class_dist_df = pd.DataFrame({
    'Clase': class_counts.index,
    'Instancias': class_counts.values,
    'Porcentaje (%)': class_pct.values
})

ratio_df = pd.DataFrame({
    'Métrica': ['Ratio mayoría/minoría'],
    'Valor': [f"{class_counts.max() / class_counts.min():.2f}:1"]
})

# Mostrar tablas EDA
display(resumen_dimensiones)
display(tabla_variables)
display(card_df)
display(missing_df)
display(nulos_obj_df)
display(constantes_ids_df)
display(class_dist_df)
display(ratio_df)

En este código podemos hacer un breve análisis de los datos de entrenamiento disponibles (columnas, variables categóricas, tipo ID, etc). Además, se identifica el tipo de problema: *clasificación binaria* y se obtiene el porcentaje de "yes" y "no" para ver la distribución de clases. En este caso, obtenemos un ~53% de "yes" y un ~47% de "no", lo que nos lleva a concluir que este dataset es equilibrado, pues el ratio es de 1.11:1, muy igualado para ambos valores.

### 1.2 — Análisis especial de `pdays` y preprocesado
Tras el análisis anterior, se observará la variable *pdays* y se tomará una decisión acerca de su uso. Posteriormente se realizará su preprocesado.

In [ ]:
import pandas as pd
from IPython.display import display

# Análisis de pdays (antes del preprocesado)
no_contactado = (df['pdays'] == -1).sum()
si_contactado = (df['pdays'] > -1).sum()

distribucion_actual_df = pd.DataFrame({
    'Estado': ['-1 (nunca contactado)', '> -1 (sí contactado)'],
    'Instancias': [no_contactado, si_contactado],
    'Porcentaje (%)': [round(no_contactado / len(df) * 100, 2), round(si_contactado / len(df) * 100, 2)]
})

pdays_contactados = df[df['pdays'] > -1]['pdays']
stats_contactados_df = pd.DataFrame({
    'Grupo': ['pdays > -1'],
    'Mínimo': [pdays_contactados.min()],
    'Máximo': [pdays_contactados.max()],
    'Media': [round(pdays_contactados.mean(), 1)]
})

# Preprocesado: crear columna 'contacted_before' con valores 1/0 y transformar pdays
preprocesado_pdays_df = pd.DataFrame({
    'Transformación': ['pdays = -1', 'pdays > -1'],
    'Resultado': ['contacted_before=0 y pdays=0', 'contacted_before=1 y pdays=días reales']
})

df['contacted_before'] = (df['pdays'] > -1).astype(int)
df['pdays'] = df['pdays'].apply(lambda x: 0 if x == -1 else x)

contacted_before_df = (
    df['contacted_before']
    .value_counts()
    .rename_axis('contacted_before')
    .reset_index(name='Instancias')
)

stats_pdays_nuevo_df = pd.DataFrame({
    'Mínimo': [df['pdays'].min()],
    'Máximo': [df['pdays'].max()],
    'Media': [round(df['pdays'].mean(), 1)],
    'Desviación estándar': [round(df['pdays'].std(), 1)]
})

# Mostrar tablas de análisis y preprocesado
display(distribucion_actual_df)
display(stats_contactados_df)
display(preprocesado_pdays_df)
display(contacted_before_df)
display(stats_pdays_nuevo_df)

Para utilizar la variable `pdays`, con un ~75% de -1 (nunca contactado) y un ~25% de valores entre 1 y 854, se ha creado una nueva variable llamada `contacted_before`, que podrá tener valores 0/1, 0 en caso de que `pdays` sea -1, y 1 en caso de que tenga otro valor. Gracias a esta nueva variable, podemos dar a `pdays` el número de días desde el último contacto (0 en caso de no contactado, y los días reales, en caso de que si haya sido contactado)

## 2 - Decidir cómo se va a realizar la evaluación

En esta fase se definen tres decisiones clave para evaluar y comparar los modelos:
1. **Métrica a usar**
2. **División train/test**
3. **Evaluación interna**

### 2.1 - Elección de la métrica

La métrica determina qué tan bien funciona el modelo. Para este proyecto, se ha decidido usar la métrica ROC-AUC.
El objetivo es predecir si un cliiente bancario suscribirá un depósito, por lo que queremos identificar qué clientes tienen una mayor probabilidad de hacerlo. Por eso es importante que podamos hacer una clasificación de los clientes por probabilidad en lugar de simplemente clasificarlos como sí/no.

#### Comparación de métricas
**Accuracy:** No es una buena métrica ya que nuestro dataset no está perfectamente equilibrado: tenemos un 53% de sí y un 47% de no, de forma que un modelo que siempre predice no tendría un 47% de acierto, lo cual no aporta nada porque es algo que ya se ve.

**F1-Score:** Es una buena métrica para el proyecto, pero no la mejor. El principal problema es que nos obliga a usar un umbral fijo, por lo que no podemos ir adaptándonos a las necesidades que tengamos. Por ejemplo, si queremos quedarnos con muchos clientes, podemos usar un umbral más bajo, mientras que si queremos pocos clientes pero asegurarlos podemos usar un umbral más alto, por lo que un umbral fijo no es óptimo.

**ROC-AUC:** Esta es la métrica escogida. Mide la capacidad del modelo para separar a los clientes con sí de los clientes con no utilizando todos los umbrales posibles. Por ejemplo, si comienza probando el umbral 0.1, la mayoría se predecirá a sí, pero habrá muchos falsos positivos. Después va subiendo el umbral, prediciendo menos sí pero siendo estos mucho más seguros (menos falsos positivos). En cada valor del umbral se calcula el porcentaje de positivos verdaderos y falsos negativos encontrados. Una vez probados todos los umbrales, el modelo traza una curva y calcula el área bajo esa curva (obtiene un número entre 0 y 1).

In [ ]:
#Métrica principal
SCORING = 'roc_auc'

### 2.2 - División train/test

Se ha dividido los datos usando **Holdout** con proporción **2/3 (train) - 1/3 (test)** y **Stratify** para asegurar que la proporción de sí/no sea la misma en train y test.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

#Creamos LabelEncoder para codificar la variable objetivo
le = LabelEncoder()

#Usamos el NIA como semilla
SEED = 100522189

X = df.drop(columns=['deposit'])
y = le.fit_transform(df['deposit'])  #0 = no, 1 = yes


print("DIVISIÓN TRAIN/TEST")

#Dividimos los datos con stratify para mantener proporciones
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=1/3,           #33.3% para test, 66.7% para train
    stratify=y,              #Mantiene la proporción de clases
    random_state=SEED,
    shuffle=True             #Mezcla aleatoria de datos
)

print()
print(f"Train: {len(X_train_full)} instancias ({len(X_train_full)/len(X)*100:.1f}%)")
print(f"Test:  {len(X_test)} instancias ({len(X_test)/len(X)*100:.1f}%)")

print()
print("Proporciones de clases con stratify")

print()
print("Dataset Original:")
original_yes = (y == 1).sum()
original_no = (y == 0).sum()
print(f"SÍ:  {original_yes:>6d} ({original_yes/len(y)*100:>5.1f}%)")
print(f"NO:  {original_no:>6d} ({original_no/len(y)*100:>5.1f}%)")

print()
print("Train:")
train_yes = (y_train_full == 1).sum()
train_no = (y_train_full == 0).sum()
print(f"YES:  {train_yes:>6d} ({train_yes/len(y_train_full)*100:>5.1f}%)")
print(f"NO:   {train_no:>6d} ({train_no/len(y_train_full)*100:>5.1f}%)")

print()
print("Conjunto TEST:")
test_yes = (y_test == 1).sum()
test_no = (y_test == 0).sum()
print(f"SÍ:  {test_yes:>6d} ({test_yes/len(y_test)*100:>5.1f}%)")
print(f"NO:  {test_no:>6d} ({test_no/len(y_test)*100:>5.1f}%)")



### 2.3 - Evaluación INNER (Cross-Validation para HPO)

Usaremos **validación cruzada** para probar diferentes valores de parámetros y elegir entre diferentes métodos (KNN, árboles, lineales, SVM) antes de usar el test set. Se hará validación cruzada con 5 folds porque es lo más recomendado y mantiene las proporciones de clases en cada fold de manera correcta.

In [ ]:
from sklearn.model_selection import StratifiedKFold

#Configuración de validación cruzada
inner_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

## 3 - KNN y Trees

En esta fase del proyecto se comenzará garantizando la robustez del modelo y limpiando los datos de entrenamiento mediante un preproceso completo usando Pipelines de Scikit-Learn. Se evitará cualquier tipo de data leakage y tras llevar a cabo el preproceso se realizarán las siguientes tareas:

1. **Selección de escalador con KNN**
2. **Evaluación de modelos base (KNN y Trees)**
3. **Conclusiones**

### 3.1 - Preproceso

A continuación se hace el preproceso con Pipelines y se deja listo para aplicar más adelante.
En este preproceso realizamos lo siguiente:

**Variables numéricas:** se utlizará la mediana para valores nulos en lugar de otras opciones como la media ya que así evitamos que valores atípicos desplacen de gran manera la media a valores poco coherentes.

**Variables ordinales:** en estas variables se observa una clara jerarquía (primary, secondary, tertiary). Por lo que se ha definido el orden unknown - primary - secondary - tertiary, con los valores 0, 1, 2 y 3, respectivamente. De esta forma damos valor a eestas variables y también asignamos uno a los valores desconocidos.

**Variables categóricas:** como estas variables no tienen un orden matemático, se ha decidido transformarlas en columnas binarias con OneHotEncoder y el handler de columnas desconocidos se ha fijado a 'ignore' para asegurarnos de que, si aparece en el test una columna no registrada en el entrenamiento, se ignore y no afecte al modelo.

Por último, todo este preproceso se guardará en un objeto ColumnTransformer y quedará a la espera de ser utilizado en próximas fases del proyecto.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

#Definimos las listas de columnas ignorando la variable objetivo
num_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
bin_cols = ['contacted_before']
cat_cols = ['job', 'marital', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
ord_cols = ['education']

#Pipeline para la variable Ordinal 'education'
ord_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('ordinal', OrdinalEncoder(categories=[['unknown', 'primary', 'secondary', 'tertiary']], 
    handle_unknown='use_encoded_value', unknown_value=-1))
])

#Pipeline para variables Categóricas
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

#Pipeline para variables Numéricas
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

#Pipeline para variables Binarias (sin escalador)
bin_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

#Guardamos todo en el ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols),
        ('bin', bin_pipeline, bin_cols),
        ('ord', ord_pipeline, ord_cols),
        ('cat', cat_pipeline, cat_cols)
    ],
    remainder='drop' #Ignoramos cualquier otra columna que no esté en las listas
)

print("Preproceso completado")

### 3.2 - Selección de escalador con KNN

Dado que KNN realiza sus predicciones en base al cálculo de distancias entre las instancias de los datos, se convierte en un algoritmo muy sensible a la escala de los valores. Es por esto que debemos aplicar técnicas de escalado y ver cuál es el escalador más apropiado para este proyecto.

Los escaladores a probar serán **MinMaxEscaler**, **StandardScaler** y **RobustEscaler**, y el procedimiento será el siguiente:

Se crearán 3 pipelines, cada uno con el preprocesador creado anteriormente, el modelo KNN y uno de los escaladores que vamos a probar. Cada alternativa se evaluará con validación cruzada interna midiendo tiempos y la métrica ROC-AUC. El escalador escogido será el que mejores métricas obtenga.

In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_validate

#Definimos los escaladores
scalers = [MinMaxScaler(), StandardScaler(), RobustScaler()]

resultados_escaladores = []

#Bucle para evaluar los escaladores
for scaler in scalers:
    #Obtenemos el nombre
    nombre = type(scaler).__name__ 
    
    #Pipeline con escalador sólo para las numericas no binarias
    num_pipeline_scaled = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('escalador', scaler)
    ])
    
    #Pipeline con escalador para las ordinales (después de OrdinalEncoder)
    ord_pipeline_scaled = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
        ('ordinal', OrdinalEncoder(categories=[['unknown', 'primary', 'secondary', 'tertiary']], 
        handle_unknown='use_encoded_value', unknown_value=-1)),
        ('escalador', scaler)
    ])
    
    #Volvemos a construir ColumnTransformer poniendo el pipeline anterior a las variables numéricas no binarias
    preprocessor_scaled = ColumnTransformer(
        transformers=[
            ('num', num_pipeline_scaled, num_cols),
            ('bin', bin_pipeline, bin_cols),
            ('ord', ord_pipeline_scaled, ord_cols),
            ('cat', cat_pipeline, cat_cols)
        ],
        remainder='drop' 
    )
    
    #Pipeline completo con KNN
    pipeline_knn = Pipeline(steps=[
        ('preprocesamiento', preprocessor_scaled),
        ('knn', KNeighborsClassifier()) # KNN con hiperparámetros por omisión
    ])
    
    #Empezamos a medir el tiempo
    start_time = time.time()

    #CV
    cv_results = cross_validate(
        pipeline_knn, 
        X_train_full, 
        y_train_full, 
        cv=inner_cv, 
        scoring='roc_auc', 
        n_jobs=-1
    )
    
    tiempo_total = time.time() - start_time
    roc_auc_medio = np.mean(cv_results['test_score'])
    
    #Guardamos los resultados en nuestra lista
    resultados_escaladores.append({
        'Escalador': nombre,
        'ROC-AUC (Media CV)': roc_auc_medio,
        'Tiempo total CV (s)': tiempo_total
    })

#Mostramos los resultados
df_resultados_knn = pd.DataFrame(resultados_escaladores).sort_values(by='ROC-AUC (Media CV)', ascending=False)
display(df_resultados_knn)

Según los resultados obtenidos, el escalador que se usará para los modelos sensibles a la magnitud de las variables es **StandardScaler**, con un ROC-AUC de 0.8782 (los tiempos varían), lo cual significa que si cogemos un cliente al azar que sí hizo el deposito y otro que no lo hizo, hay un 87.8% de probabilidad de que nuestro modelo de una mayor puntuación a ese cliente que sí lo hizo.



### 3.3 - Evaluación (Hiperparámetros por omisión)

En la selección de escalador para KNN ya se realiza una evaluación con StandardScaler con hiperparámetros por omisión, de manera que en esta sección únicamente se evaluará usando Árboles de Decisión. Estos se basan en tomar ciertas decisiones (>30?, <144?) de forma que la magnitud de las variables no afecta, por lo que el pipeline para su evaluación no usarSstEscaler (no sirve de nada aquí).

In [ ]:
import time
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate

#Construimos el pipeline
pipeline_tree_nohpo = Pipeline(steps=[
    ('preprocesamiento', preprocessor),
    ('tree', DecisionTreeClassifier(random_state=89)) 
])

#Empezamos a medir el tiempo
start_time = time.time()

#CV
cv_results_tree = cross_validate(
    pipeline_tree_nohpo, 
    X_train_full, 
    y_train_full, 
    cv=inner_cv, 
    scoring='roc_auc', 
    n_jobs=-1
)

tiempo_tree = time.time() - start_time
roc_auc_tree = np.mean(cv_results_tree['test_score'])

#Guardamos los resultados en un DataFrame
resultados_tree = [{
    'Modelo': 'Decision Tree',
    'ROC-AUC (Media CV)': roc_auc_tree,
    'Tiempo total CV (s)': tiempo_tree
}]

df_resultados_tree = pd.DataFrame(resultados_tree)
display(df_resultados_tree)

### 3.4 - Árboles poco profundos 

Una gran ventaja de los Árboles de Decisión es que son altamente interpretables. Para poder comprender cómo nuestro modelo toma sus decisiones y cuáles son las variables más predicitvas de todo nuestro conjunto de datos, vamos a entrenar un árbol de poca profundidad (en este caso 3). Se ha utilizado el preprocesador del apartado 3.1 y se ha configurado el modelo para utilizar la entropía como criterio para evaluar las divisiones (en HPO se valorará otra alternativa).

Tras realizar el entrenamiento, se generará un diagrama visual que permitirá realizar un análisis más sencillo de los resultados obtenidos.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline

#Construimos el pipeline con un árbol de profundidad 3
pipeline_tree = Pipeline(steps=[
    ('preprocesamiento', preprocessor),
    ('tree', DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=SEED))
])

#Usamos el conjunto de entrenamiento
pipeline_tree.fit(X_train_full, y_train_full)

#Sacamos el modelo de árbol entrenado
tree_model = pipeline_tree.named_steps['tree']

#Sacamos los nombres de las columnas generadas por el preprocesador
feature_names = pipeline_tree.named_steps['preprocesamiento'].get_feature_names_out()

#Dibujamos el diagrama del árbol
plt.figure(figsize=(22, 12))
plot_tree(
    tree_model,
    feature_names=feature_names,
    class_names=['No Suscribe', 'Sí Suscribe'],
    filled=True,
    rounded=True,
    fontsize=10
)

plt.title("Diagrama del Árbol de profundidad 3", fontsize=16)
plt.show()

Observando el diagrama generado, podemos llegar a las siguientes conclusiones:

**num_duration:** la duración de la llamada es el factor más importante de todo el conjunto de datos. El nodo raíz divide a los clientes en función de si la llamada duró más o menos de 206,5 segundos, por lo que sabemos que el tiempo que el cliente pase hablando en la llamada es una clara muestra de interés de cara a suscribir el depósito. Además, esta misma variable es tan dominante que el árbol la vuelve a utilizar en el segundo nivel para seguir dividiendo a los clientes.

**Nodos naranjas:** en caso de que la duración de la llamada sea menor a 206,5 segundos, el árbol tiene una fuerte tendencia a la clase "No suscribe", que se representa en nodos naranjas. En estos casos la variable cat__poutcome_success es esencial para saber que si el cliente no ha respondido positivamente en una campaña previa, aumenta aún más la probabilidad de "No suscribe". También se vuelve a evaluar la duración de la llamada.

**Nodos azules:** en caso de que la duración de la llamada sea mayor a 206,5 segundos, el modelo sabe que el cliente es muy posiblemente de la clase "Sí suscribe", que se representa en nodos azules. Se vuelve a evaluar la duración, (esta vez con 472,5 segundos). Si la duración es menor, se revisa la variable cat__contact_unknown para verificar si el contacto es desconocido, lo cual aumentaría la probabilidad de "No suscribe". Por otro lado, si la llamada ha superado la cota anterior, se volverá a evaluar una vez más (con 668,5 segundos). Tanto si duró más como si duró menos la probabilidad de "Sí suscribe" es muy alta, pero cuanto mayor sea num__duration, más probable es.

En conclusión, este modelo de árbol nos muestra que los factores más importantes a tener en cuenta son la duración de las llamadas con clientes, el método de contacto y el éxito obtenido en campañas previas.

### 3.5 - HPO

Una vez realizada la evaluación con hiperparámetros por omisión, el siguiente paso es optimizarlos. Para ello, utilizaremos GridSearchCV y RandomizedSearchCV.

Comenzaremos haciendo **HPO en KNN**, optimizando el número de vecinos, el peso de las distancias y la métrica de cálculo, todo ello con el escalador ya seleccionado.



In [ ]:
import warnings
import numpy as np
import time
import pandas as pd 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

print("HPO KNN")
start_time = time.time()

#Reconstruimos los pipelines de variables numéricas y ordinales con el scaler
num_pipeline_opt = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('escalador', StandardScaler())
])

ord_pipeline_opt = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('ordinal', OrdinalEncoder(
        categories=[['unknown', 'primary', 'secondary', 'tertiary']], 
        handle_unknown='use_encoded_value', 
        unknown_value=-1
    )),
    ('escalador', StandardScaler())
])

#Juntamos todo en ColumnTransformer
preprocessor_opt = ColumnTransformer(
    transformers=[
        ('num', num_pipeline_opt, num_cols),
        ('ord', ord_pipeline_opt, ord_cols),
        ('bin', bin_pipeline, bin_cols),
        ('cat', cat_pipeline, cat_cols)
    ],
    remainder='drop' 
)

#Construimos el pipeline base de KNN
pipeline_knn = Pipeline(steps=[
    ('preprocesamiento', preprocessor_opt),
    ('knn', KNeighborsClassifier())
])

#Definimos el espacio de búsqueda para GridSearchCV
param_grid_knn = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 15, 21, 31, 41],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

#Ejecutamos GridSearchCV
grid_search_knn = GridSearchCV(
    pipeline_knn,
    param_grid_knn,
    cv=inner_cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

grid_search_knn.fit(X_train_full, y_train_full)

tiempo_hpo_knn = time.time() - start_time

#Extraemos los resultados
estudio_knn_best_value = grid_search_knn.best_score_
estudio_knn_best_params = grid_search_knn.best_params_

#Mostramos datos de búsqueda
results_df = pd.DataFrame(grid_search_knn.cv_results_)
best_idx = grid_search_knn.best_index_

print(f"\nHPO completado en {tiempo_hpo_knn:.2f} s")
print(f"Mejor ROC-AUC (Media CV): {estudio_knn_best_value:.4f}")
print(f"Mejores hiperparámetros: {estudio_knn_best_params}")
print(f"\nCombinaciones probadas: {len(results_df)}")

#Gráficos
#Calcular importancia
def calcular_importancia_parametros(results_df, param_names):
    """Calcula la importancia de cada parámetro basándose en varianza de ROC-AUC"""
    importancia = {}
    for param in param_names:
        param_values = {}
        for idx, row in results_df.iterrows():
            val = row['params'].get(param, 'unknown')
            if val not in param_values:
                param_values[val] = []
            param_values[val].append(row['mean_test_score'])
        
        medias = [np.mean(scores) for scores in param_values.values()]
        
        if len(medias) > 0:
            importancia[param] = max(medias) - min(medias)
        else:
            importancia[param] = 0
    
    return importancia

#Calcular importancia KNN
knn_params = ['knn__n_neighbors', 'knn__weights', 'knn__p']
knn_importance = calcular_importancia_parametros(results_df, knn_params)
knn_importance_normalized = {k.replace('knn__', ''): v/max(knn_importance.values()) if max(knn_importance.values()) > 0 else 0 
                             for k, v in knn_importance.items()}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

#KNN - Historial
ax1 = axes[0]
knn_results = pd.DataFrame(grid_search_knn.cv_results_)
ax1.plot(range(len(knn_results)), knn_results['mean_test_score'], 'o-', color='#1f77b4', alpha=0.7, linewidth=2)
ax1.axhline(y=knn_results['mean_test_score'].max(), color='r', linestyle='--', alpha=0.5, label=f'Máximo: {knn_results["mean_test_score"].max():.4f}')
ax1.set_xlabel('Evaluación', fontweight='bold')
ax1.set_ylabel('ROC-AUC', fontweight='bold')
ax1.set_title('Historial de Búsqueda - KNN (GridSearchCV)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

#KNN - Importancia de parámetros
ax2 = axes[1]
knn_names = list(knn_importance_normalized.keys())
knn_values = list(knn_importance_normalized.values())
bars = ax2.barh(knn_names, knn_values, color='#1f77b4', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Importancia Normalizada', fontweight='bold')
ax2.set_title('Importancia de Hiperparámetros - KNN', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 1.1)
for i, (bar, val) in enumerate(zip(bars, knn_values)):
    ax2.text(val + 0.02, i, f'{val:.3f}', va='center', fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

Al observar los gráficos generados y los resultados obtenidos, llegamos a las siguientes conclusiones:
1. **Historial de Optimización:** El modelo va subiendo paulatinamente hasta alcanzar la barrera de los 0.89, donde se estabiliza un poco más.
2. **Importancia de los Hiperparámetros:** El gráfico de importancia revela que el número de vecinos es el factor crítico que define el éxito del modelo KNN en este conjunto de datos. Por otro lado, cambiar la métrica de cálculo entre Manhattan y Euclídea o los pesos tiene un impacto menor, pero no se debe dejar de lado. Podemos concluir que lo más importante es tener un número de vecinos elevado, concretamente 21 si queremos obtener los mejores resultados.

A continuación, se realizará **HPO en Árboles de Decisión**, optimizando el criterio entre gini y entropía, los límites y la profundidad del árbol.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats
import numpy as np

print("HPO ÁRBOLES")
start_time = time.time()

#Construimos el pipeline base con el preprocesador
pipeline_tree = Pipeline(steps=[
    ('preprocesamiento', preprocessor),
    ('tree', DecisionTreeClassifier(random_state=SEED))
])

#Definimos el espacio de búsqueda para RandomizedSearchCV
param_dist_tree = {
    'tree__criterion': ['gini', 'entropy'],
    'tree__max_depth': stats.randint(2, 20),
    'tree__min_samples_split': stats.randint(2, 100),
    'tree__min_samples_leaf': stats.randint(1, 50)
}

#Ejecutamos RandomizedSearchCV
random_search_tree = RandomizedSearchCV(
    pipeline_tree,
    param_dist_tree,
    n_iter=50,
    cv=inner_cv,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=SEED,
    verbose=0
)

random_search_tree.fit(X_train_full, y_train_full)

tiempo_hpo_tree = time.time() - start_time

#Extraemos los resultados
estudio_tree_best_value = random_search_tree.best_score_
estudio_tree_best_params = random_search_tree.best_params_

#Mostramos datos de búsqueda
results_df_tree = pd.DataFrame(random_search_tree.cv_results_)
best_idx_tree = random_search_tree.best_index_

print(f"\nHPO completado en {tiempo_hpo_tree:.2f} segundos.")
print(f"Mejor ROC-AUC (Media CV): {estudio_tree_best_value:.4f}")
print(f"Mejores hiperparámetros:  {estudio_tree_best_params}")
print(f"\nCombinaciones probadas: {len(results_df_tree)}")

#Gráficos
#Calcular importancia Árbol
tree_params = ['tree__criterion', 'tree__max_depth', 'tree__min_samples_split', 'tree__min_samples_leaf']
tree_importance = calcular_importancia_parametros(results_df_tree, tree_params)
tree_importance_normalized = {k.replace('tree__', ''): v/max(tree_importance.values()) if max(tree_importance.values()) > 0 else 0 
                              for k, v in tree_importance.items()}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

#Árbol - Historial
ax1 = axes[0]
tree_results = pd.DataFrame(random_search_tree.cv_results_)
ax1.plot(range(len(tree_results)), tree_results['mean_test_score'], 'o-', color='#ff7f0e', alpha=0.7, linewidth=2)
ax1.axhline(y=tree_results['mean_test_score'].max(), color='r', linestyle='--', alpha=0.5, label=f'Máximo: {tree_results["mean_test_score"].max():.4f}')
ax1.set_xlabel('Evaluación', fontweight='bold')
ax1.set_ylabel('ROC-AUC', fontweight='bold')
ax1.set_title('Historial de Búsqueda - Árbol de Decisión (RandomizedSearchCV)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

#Árbol - Importancia de parámetros
ax2 = axes[1]
tree_names = list(tree_importance_normalized.keys())
tree_values = list(tree_importance_normalized.values())
bars = ax2.barh(tree_names, tree_values, color='#ff7f0e', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Importancia Normalizada', fontweight='bold')
ax2.set_title('Importancia de Hiperparámetros - Árbol de Decisión', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 1.1)
for i, (bar, val) in enumerate(zip(bars, tree_values)):
    ax2.text(val + 0.02, i, f'{val:.3f}', va='center', fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

Al observar los gráficos generados y los resultados obtenidos, llegamos a las siguientes conclusiones:
1. **Historial de Optimización:** El modelo tiene unos resultados variables a lo largo del proceso de optimización, llegando al pico de ROC-AUC 0.89, similar a KNN.
2. **Importancia de los Hiperparámetros:** A excepción del criterio, todos los hiperparámetros son fundamentales para la optimización de los árboles de decisión, por lo que es fundamental prestar atención a los valores asignados a la profundidad o el número de elementos para cada lugar del árbol.

### 3.6 - Análisis de tiempos y efecto de los hiperparámetros

A continuación se realizará una comparación entre los modelos con hiperparámetros por omisión y los modelos con HPO. El objetivo es medir la mejora de resultados de los modelos con HPO y ver el coste computacional extra de estos.

Se obtendrán métricas de:
1. **ROC-AUC**
2. **Tiempos de entrenamiento**


In [ ]:
import pandas as pd
import time
from sklearn.dummy import DummyClassifier

#Medir tiempo de evaluación base de KNN
start = time.time()
cv_knn_base = cross_validate(Pipeline(steps=[
    ('preprocesamiento', preprocessor_opt),
    ('knn', KNeighborsClassifier(n_neighbors=5))
]), X_train_full, y_train_full, cv=inner_cv, scoring='roc_auc', n_jobs=2)
tiempo_knn_base = time.time() - start
roc_auc_knn_base = cv_knn_base['test_score'].mean()

#Medir tiempo de evaluación base del Árbol
start = time.time()
cv_tree_base = cross_validate(Pipeline(steps=[
    ('preprocesamiento', preprocessor),
    ('tree', DecisionTreeClassifier(random_state=SEED))
]), X_train_full, y_train_full, cv=inner_cv, scoring='roc_auc', n_jobs=2)
tiempo_tree_base = time.time() - start
roc_auc_tree_base = cv_tree_base['test_score'].mean()

#Medir tiempo de Dummy Classifier (Most Frequent)
start = time.time()
cv_dummy_mf = cross_validate(Pipeline(steps=[
    ('preprocesamiento', preprocessor),
    ('dummy', DummyClassifier(strategy='most_frequent'))
]), X_train_full, y_train_full, cv=inner_cv, scoring='roc_auc', n_jobs=2)
tiempo_dummy_mf = time.time() - start
roc_auc_dummy_mf = cv_dummy_mf['test_score'].mean()

#Medir tiempo de Dummy Classifier (Stratified)
start = time.time()
cv_dummy_strat = cross_validate(Pipeline(steps=[
    ('preprocesamiento', preprocessor),
    ('dummy', DummyClassifier(strategy='stratified', random_state=SEED))
]), X_train_full, y_train_full, cv=inner_cv, scoring='roc_auc', n_jobs=2)
tiempo_dummy_strat = time.time() - start
roc_auc_dummy_strat = cv_dummy_strat['test_score'].mean()

#Crear tabla comparativa
df_comparativa = pd.DataFrame({
    'Modelo': ['Dummy (Más Frecuente)', 'Dummy (Estratificado)', 'KNN (Sin HPO)', 'KNN (Con HPO)', 'Árbol (Sin HPO)', 'Árbol (Con HPO)'],
    'ROC-AUC': [roc_auc_dummy_mf, roc_auc_dummy_strat, roc_auc_knn_base, estudio_knn_best_value, roc_auc_tree_base, estudio_tree_best_value],
    'Tiempo (s)': [tiempo_dummy_mf, tiempo_dummy_strat, tiempo_knn_base, tiempo_hpo_knn, tiempo_tree_base, tiempo_hpo_tree],
    'Método': ['Baseline', 'Baseline', 'Base', 'GridSearchCV', 'Base', 'RandomizedSearchCV'],
    'Evaluaciones': [1, 1, 1, 36, 1, 50]
})

display(df_comparativa.round(4))

Analizando la tabla comparativa llegamos a las siguientes conclusiones:

1. **ROC-AUC:** tanto KNN como árboles logran una mejora en su calidad predictiva. KNN obtiene una mejora no tan notable, mientras que árboles se beneficia de gran manera del HPO. Ambos modelos llegan casi al 0.90 de ROC-AUC y, como es lógico, superan con creces a los modelos Dummy, que obtienen unos resutlados muy pobres.
2. **Coste Computacional:** la optimización de ambos modelos supone un aumento de tiempo de un par de segundos (los tiempos pueden variar un poco). Esto no supone un problema, ya que se trata de un breve aumento de tiempo frente a una importante mejoría que en un entorno real puede tener un gran impacto. También se aprecia un aumento de tiempo en comparación a los modelos Dummy, lo cual es normal y positivo considerando el gran aumento de calidad predictiva.

## 4 - Modelos Lineales y SVMs

En esta fase se evaluarán **Modelos Lineales** y **Máquinas de Vectores de Soporte**. En el caso de modelos lineales, se usará regresión logística porque sus coeficientes indican el efecto de cada variable sobre la probabilidad de contratar el depósito. Los positivos significan una mayor probabilidad de éxito, y los negativos una menor.

Se comenzará evaluando el rendimiento de estos modelos con hiperparámetros por omisión. Después, se realizará HPO y se volverá a evaluar. Por último se extraeran conclusiones al respecto.

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import cross_validate

#Reconstruimos el preprocesador con StandardScaler
num_pipeline_adv = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), 
    ('escalador', StandardScaler())
])

ord_pipeline_adv = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('ordinal', OrdinalEncoder(categories=[['unknown', 'primary', 'secondary', 'tertiary']], handle_unknown='use_encoded_value', unknown_value=-1)),
    ('escalador', StandardScaler())
])

preprocessor_adv = ColumnTransformer(
    transformers=[
        ('num', num_pipeline_adv, num_cols),
        ('ord', ord_pipeline_adv, ord_cols),
        ('bin', bin_pipeline, bin_cols),
        ('cat', cat_pipeline, cat_cols)
    ], remainder='drop'
)

#Definimos los modelos
modelos_avanzados = {
    #Modelo Lineal sin L1
    'Regresión Logística (Sin L1)': LogisticRegression(max_iter=1000, random_state=SEED),
    
    #Modelo Lineal con L1
    'Regresión Logística (Con L1)': LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=SEED),
    
    #SVM
    'SVM': SVC(random_state=SEED)
}

resultados_avanzados = []

#CV
for nombre, modelo in modelos_avanzados.items():
    pipeline_adv = Pipeline(steps=[
        ('preprocesamiento', preprocessor_adv),
        ('clasificador', modelo)
    ])
    
    start_time = time.time()
    
    cv_results = cross_validate(
        pipeline_adv, 
        X_train_full, 
        y_train_full, 
        cv=inner_cv, 
        scoring='roc_auc', 
        n_jobs=-1
    )
    
    tiempo_total = time.time() - start_time
    roc_auc_medio = np.mean(cv_results['test_score'])
    
    resultados_avanzados.append({
        'Modelo': nombre,
        'ROC-AUC (Media CV)': roc_auc_medio,
        'Tiempo (s)': tiempo_total
    })
    

#Mostrar en formato tabla
print("TABLA DE RESULTADOS")
df_resultados_adv = pd.DataFrame(resultados_avanzados).sort_values(by='ROC-AUC (Media CV)', ascending=False)
display(df_resultados_adv)

Al observar los resultados llegamos a las siguientes conclusiones:

**Aumento de calidad:** los modelos lineales muestran un excelente ROC-AUC de ~0.905. Se trata de un resultado que, sin haber optimizado aún los hiperparámetros, ya supera los resultados óptimos logrados en KNN y árboles. No se observa una gran diferencia en el uso de L1.

**SVM:** la Máquina de Vectores de Soporte obtiene el mejor resultado hasta el momento, con un ROC-AUC de 0.92, superando con creces todos los modelos anteriores, sin haber realizado HPO aún. Sin embargo, este excelente resultado muestra el aumento de coste computacional.

### 4.1 - HPO modelos lineales

A continuación, se realiza el ajuste de hiperparámetros de modelos lineales.

En Regresión Logística, el hiperparámetro más importante a optimizar es **C**. Un valor bajo de **C** fuerza al modelo a ser más simple y evita el sobreajuste, mientras que un valor más elevado permite que el modelo se ajuste más a los datos de entrenamiento.


In [ ]:
import warnings
import numpy as np
import time
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings('ignore')

#HPO Regresión Logística L2
start_time_lr_l2 = time.time()

#Construimos el pipeline base para L2
pipeline_lr_l2 = Pipeline(steps=[
    ('preprocesamiento', preprocessor_adv),
    ('clasificador', LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000, random_state=SEED))
])

#Espacio de búsqueda
C_values_l2 = np.logspace(-4, 2, 50)
param_grid_lr_l2 = {'clasificador__C': C_values_l2}

#Ejecutamos GridSearchCV
grid_search_lr_l2 = GridSearchCV(
    pipeline_lr_l2,
    param_grid_lr_l2,
    cv=inner_cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

grid_search_lr_l2.fit(X_train_full, y_train_full)
tiempo_hpo_lr_l2 = time.time() - start_time_lr_l2

estudio_lr_l2_best_value = grid_search_lr_l2.best_score_
estudio_lr_l2_best_params = grid_search_lr_l2.best_params_
results_l2 = pd.DataFrame(grid_search_lr_l2.cv_results_)

#HPO Regresión Logística L1
start_time_lr_l1 = time.time()

#Construimos el pipeline base para L1
pipeline_lr_l1 = Pipeline(steps=[
    ('preprocesamiento', preprocessor_adv),
    ('clasificador', LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=SEED))
])

#Espacio de búsqueda
C_values_l1 = np.logspace(-4, 2, 50)
param_grid_lr_l1 = {'clasificador__C': C_values_l1}

#Ejecutamos GridSearchCV
grid_search_lr_l1 = GridSearchCV(
    pipeline_lr_l1,
    param_grid_lr_l1,
    cv=inner_cv,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

grid_search_lr_l1.fit(X_train_full, y_train_full)
tiempo_hpo_lr_l1 = time.time() - start_time_lr_l1

estudio_lr_l1_best_value = grid_search_lr_l1.best_score_
estudio_lr_l1_best_params = grid_search_lr_l1.best_params_
results_l1 = pd.DataFrame(grid_search_lr_l1.cv_results_)


#Tabla de resultados
print(" RESULTADOS HPO")

df_lr_resultados = pd.DataFrame({
    'Regularización': ['L2', 'L1'],
    'ROC-AUC': [estudio_lr_l2_best_value, estudio_lr_l1_best_value],
    'C Óptimo': [estudio_lr_l2_best_params['clasificador__C'], estudio_lr_l1_best_params['clasificador__C']],
    'Tiempo HPO (s)': [tiempo_hpo_lr_l2, tiempo_hpo_lr_l1],
    'Evaluaciones': [len(results_l2), len(results_l1)]
})

display(df_lr_resultados)

#Gráficos
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

#L2 - Historial
ax1 = axes[0]
ax1.plot(range(len(results_l2)), results_l2['mean_test_score'], 'o-', color='steelblue', alpha=0.7, linewidth=2)
ax1.axhline(y=results_l2['mean_test_score'].max(), color='r', linestyle='--', alpha=0.5, label=f'Máximo: {results_l2["mean_test_score"].max():.4f}')
ax1.set_xlabel('Iteración', fontweight='bold')
ax1.set_ylabel('ROC-AUC', fontweight='bold')
ax1.set_title('Historial de Optimización - Regresión Logística (L2)', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

#L1 - Historial
ax2 = axes[1]
ax2.plot(range(len(results_l1)), results_l1['mean_test_score'], 'o-', color='coral', alpha=0.7, linewidth=2)
ax2.axhline(y=results_l1['mean_test_score'].max(), color='r', linestyle='--', alpha=0.5, label=f'Máximo: {results_l1["mean_test_score"].max():.4f}')
ax2.set_xlabel('Iteración', fontweight='bold')
ax2.set_ylabel('ROC-AUC', fontweight='bold')
ax2.set_title('Historial de Optimización - Regresión Logística (L1)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

Tras observar los gráficos y los resultados obtenidos, llegamos a las siguientes conclusiones:

**Impacto del parámetro C:** Si aplicamos una regularización demasiado fuerte, nuestro modelo se vuelve demasiado simple y los resultados caen. Los mejores valores son los más equilibrados.
**L1 y L2:** Tanto con L1 como con L2, la regresión logística logran unos máximos muy similares (la trayectoria inicial es más suave en L2 y más brusca en L1). Dado que L1 se caracteriza por eliminar variables que no son útiles y apenas supera a L2 por ~0.0003 de ROC-AUC, podemos deducir que todas nuestras variables tienen importancia a la hora de realizar predicciones.

### 4.2 - HPO para SVM

A continuación se realiza la optimización de hiperparámetros para las Máquinas de Vectores de Soporte. Los parámetros a optimizar serán la **regularización**, el **kernel**, el coeficiente **gamma** y el **grado**.


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV
from scipy import stats
from IPython.display import display

print("\nHPO SVM")

start_time_svm = time.time()

#Construimos el pipeline base
pipeline_svm = Pipeline(steps=[
    ('preprocesamiento', preprocessor_adv),
    ('clasificador', SVC(max_iter=15000, random_state=SEED))
    ])

#Definimos el espacio de búsqueda para RandomizedSearchCV
param_dist_svm = {
    'clasificador__C': stats.loguniform(0.1, 1000),
    'clasificador__kernel': ['linear', 'rbf', 'poly'],
    'clasificador__gamma': ['scale', 'auto'],
    'clasificador__degree': [2, 3, 4]
}

#Ejecutamos RandomizedSearchCV
random_search_svm = RandomizedSearchCV(
    pipeline_svm,
    param_dist_svm,
    n_iter=50,
    cv=inner_cv,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=SEED,
    verbose=0
)

random_search_svm.fit(X_train_full, y_train_full)

tiempo_hpo_svm = time.time() - start_time_svm

#Extraemos los resultados
estudio_svm_best_value = random_search_svm.best_score_
estudio_svm_best_params = random_search_svm.best_params_
results_svm = pd.DataFrame(random_search_svm.cv_results_)

#Tabla de resultados
resultados_hpo_svm = pd.DataFrame({
    'ROC-AUC': [estudio_svm_best_value],
    'Kernel Óptimo': [estudio_svm_best_params['clasificador__kernel']],
    'C Óptimo': [f"{estudio_svm_best_params['clasificador__C']:.6f}"],
    'Gamma': [estudio_svm_best_params['clasificador__gamma']],
    'Degree': [estudio_svm_best_params['clasificador__degree']],
    'Tiempo HPO (s)': [round(tiempo_hpo_svm, 2)]
})

resultados_hpo_svm = resultados_hpo_svm.round(4)
display(resultados_hpo_svm)

#Función para calcular importancia
def calcular_importancia_parametros(results_df, param_names):
    """Calcula la importancia de cada parámetro basándose en varianza de ROC-AUC"""
    importancia = {}
    for param in param_names:
        param_values = {}
        for idx, row in results_df.iterrows():
            val = row['params'].get(param, 'unknown')
            if val not in param_values:
                param_values[val] = []
            param_values[val].append(row['mean_test_score'])
        
        medias = [np.mean(scores) for scores in param_values.values()]
        
        if len(medias) > 0:
            importancia[param] = max(medias) - min(medias)
        else:
            importancia[param] = 0
    
    return importancia

#Calcular importancia SVM
svm_params = ['clasificador__C', 'clasificador__kernel', 'clasificador__gamma', 'clasificador__degree']
svm_importance = calcular_importancia_parametros(results_svm, svm_params)
svm_importance_normalized = {k.replace('clasificador__', ''): v/max(svm_importance.values()) if max(svm_importance.values()) > 0 else 0 
                             for k, v in svm_importance.items()}

#Gráficos
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

#Historial de optimización
ax1 = axes[0]
ax1.plot(range(len(results_svm)), results_svm['mean_test_score'], 'o-', color='seagreen', alpha=0.7, linewidth=2)
ax1.axhline(y=results_svm['mean_test_score'].max(), color='r', linestyle='--', alpha=0.5, label=f'Máximo: {results_svm["mean_test_score"].max():.4f}')
ax1.set_xlabel('Iteración', fontweight='bold')
ax1.set_ylabel('ROC-AUC', fontweight='bold')
ax1.set_title('Historial de Optimización - SVM', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend()

#Importancia de parámetros
ax2 = axes[1]
svm_names = list(svm_importance_normalized.keys())
svm_values = list(svm_importance_normalized.values())
bars = ax2.barh(svm_names, svm_values, color='seagreen', alpha=0.7, edgecolor='black')
ax2.set_xlabel('Importancia Normalizada', fontweight='bold')
ax2.set_title('Importancia de Hiperparámetros - SVM', fontsize=12, fontweight='bold')
ax2.set_xlim(0, 1.1)
for i, (bar, val) in enumerate(zip(bars, svm_values)):
    ax2.text(val + 0.02, i, f'{val:.3f}', va='center', fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

Tras observar los gráficos y los resultados obtenidos, llegamos a las siguientes conclusiones:

**El modelo base gana:** El SVM con hiperparámetros por omisión (0.920) ha superado ligeramente al mejor modelo encontrado durante la optimización (0.919).  Esto sucede porque la búsqueda realizada ha encontrado como mejor valor de C uno peor al predeterminado por milésimas, lo cual deja esta mínima diferencia. El kernel y gamma utilizado en ambos es el mismo, y el grado se ignora también en ambos. Lo positivo que podemos extraer de esto es que la configuración base tiene unos resultados excepcionales con un costo computacional muy reducido, por lo que puede ser más productivo utilizar esa en lugar de la de HPO.

## 5 - Resultados y modelo final
### 5.1 - Seleccion del mejor modelo
Tras evaluar los distintos modelos utilizando CV inner, se comparan los resultados obtenidos en términos de ROC-AUC y el coste computacional asociado.

Los resultados muestran que:
- El rendimiento de KNN es inyferlior al de otros mod l os os a pesaar de su mejora con ajuste de hiperparámetros.
- El rregresión logística obtiene resultados muy competitivos (~0.90 ROC-AUC) comparandolo con el modelo SVM, además cuentan con la ventaje de ser interpretables.
- El modelo SVM con kernel RBF alcanza el mejor rendimiento (~0.92 ROC-AUC), aunque con mayor coste computacional.

Dado que el objetivo principal es maximizar la capacidad predictiva del modelo, se ha seleccionado el modelo SVM como la mejor alternativa.

### 5.2 - Estimación del rendimiento a futuro 
Una vez seleccionado el mejor modelo utilizando la evaluación inner, se pasa con la estimación de rendimiento a futuro del modelo seleccionado utilizando el cojunto de test. 

Este conjunto de test se utiliza únicamente en este punto, de esta forma evitamos sesgos en la selección del modelo. También se utiliza la matriz de confusión para comprender mejor los errores que comete el modelo en el conjunto de test.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

#Modelo final seleccionado, SVM base
modelo_final_test = Pipeline(steps=[
    ('preprocesamiento', preprocessor_adv),
    ('clasificador', SVC(random_state=SEED))
])

#Entrenamiento sobre train
modelo_final_test.fit(X_train_full, y_train_full)
y_score_test = modelo_final_test.decision_function(X_test)

#ROC-AUC
roc_auc_test = roc_auc_score(y_test, y_score_test)

print(f"ROC-AUC en test: {roc_auc_test:.4f}")

#Predicciones en test
y_pred_test = modelo_final_test.predict(X_test)

#Matriz de confusión
cm = confusion_matrix(y_test, y_pred_test)
labels = ['No suscribe', 'Sí suscribe']

#Porcentajes por fila
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

#Texto de cada celda
annot = np.empty_like(cm).astype(str)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        annot[i, j] = f"{cm[i, j]}\n({cm_pct[i, j]:.1f}%)"

#Dibujar
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap='Purples')

#Ejes
ax.set_xticks(np.arange(len(labels)))
ax.set_yticks(np.arange(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)

ax.set_xlabel("Predicción")
ax.set_ylabel("Valor real")
ax.set_title("Matriz de confusión en test")

#Anotaciones
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, annot[i, j],
                ha="center", va="center",
                color="white" if cm[i, j] > cm.max()/2 else "black",
                fontsize=12)

#Barra de color
fig.colorbar(im)

plt.tight_layout()
plt.show()

El modelo SVM seleccionado alcanza un valor de ROC-AUC en el conjunto de test de **0.9196**.

Este resultado es consistente con el obtenido durante la evaluación inner, lo que sugiere que el modelo generaliza adecuadamente y diferencia adecuadamente entre clientes que suscriben el depósito y clientes que no lo hacen.

La matriz de confusión permite analizar el comportamiento del modelo en el conjunto de test tanto en valores absolutos como en porcentajes.

Se observa que:

- El modelo clasifica correctamente **1592 clientes que no suscriben el depósito**, lo que representa un **82.6%** de esta clase.
- Además, identifica correctamente **1512 clientes que sí suscriben el depósito**, alcanzando un **86.9%** de aciertos en esta clase.

En cuanto a los errores:

- Se producen **335 falsos positivos** (17.4%), es decir, clientes que el modelo predice como interesados cuando en realidad no lo están.
- Se producen **228 falsos negativos** (13.1%), es decir, clientes que sí estaban interesados pero no han sido detectados por el modelo.

En general, el modelo presenta un comportamiento equilibrado entre ambas clases, con una tasa de acierto ligeramente superior en la detección de clientes que sí suscriben el depósito.


### 5.3 - Entrenamiento del modelo final

Una vez estimado el rendimiento futuro del modelo mediante el conjunto de test, se entrena de nuevo el modelo final utilizando todos los datos disponibles, con esto aprovechamos toda la información del conjunto original.

In [ ]:
import pandas as pd
import numpy as np

#Unimos train y test para entrenar el modelo definitivo
X_total = pd.concat([X_train_full, X_test], axis=0)
y_total = np.concatenate([y_train_full, y_test])

modelo_final = Pipeline(steps=[
    ('preprocesamiento', preprocessor_adv),
    ('clasificador', SVC(random_state=SEED, max_iter=-1))
])

modelo_final.fit(X_total, y_total)

print("Entrenamiento del modelo final terminado")

### 5.4 - Guardado del modelo

El modelo final se guarda en un fichero para poder reutilizarlo posteriormente tanto en la generación de predicciones para competición como en el despliegue mediante Streamlit.

In [ ]:
import joblib

joblib.dump(modelo_final, "modelo_final.joblib")
joblib.dump(le, "label_encoder.pkl")
print("Modelo guardado")

### Uso de IA
Se ha utilizado la IA para ayudarnos en las siguientes tareas:
- Corregir algunos errores de sintaxis
- Depuración de codigo 
- Asistencia en la generacion de graficos.

### Demostración del despliegue con Streamlit

A continuación se crean dos nuevas instancias de clientes y se obtienen sus predicciones utilizando la pipeline del modelo final.

Posteriormente, estos mismos valores se introducirán manualmente en la aplicación Streamlit para comprobar que las predicciones coinciden.

In [ ]:
import pandas as pd

# Dos nuevas instancias de ejemplo
clientes_demo = pd.DataFrame([
    {
        "age": 35,
        "job": "technician",
        "marital": "single",
        "education": "tertiary",
        "default": "no",
        "balance": 1500,
        "housing": "yes",
        "loan": "no",
        "contact": "cellular",
        "day": 12,
        "month": "may",
        "duration": 320,
        "campaign": 1,
        "pdays": -1,
        "previous": 0,
        "poutcome": "unknown"
    },
    {
    "age": 42,
    "job": "management",
    "marital": "married",
    "education": "tertiary",
    "default": "no",
    "balance": 3000,
    "housing": "no",
    "loan": "no",
    "contact": "cellular",
    "day": 10,
    "month": "may",
    "duration": 400,
    "campaign": 1,
    "pdays": 10,
    "previous": 2,
    "poutcome": "success"
}
])

#Guardamos una copia con los valores originales para mostrarla
clientes_demo_original = clientes_demo.copy()

#Aplicamos la misma transformación usada en entrenamiento para pdays
clientes_demo["contacted_before"] = (clientes_demo["pdays"] > -1).astype(int)
clientes_demo["pdays"] = clientes_demo["pdays"].apply(lambda x: 0 if x == -1 else x)

#Predicciones del modelo final
pred_demo = modelo_final.predict(clientes_demo)
pred_demo_labels = le.inverse_transform(pred_demo)

#Tabla final para comparar luego con Streamlit
resultado_demo = clientes_demo_original.copy()
resultado_demo["prediccion_modelo"] = pred_demo_labels

print("Instancias de prueba para comparar con Streamlit:")
display(resultado_demo)

## 6 - Tarea de elección abierta: Gradient Boosting

Hemos decido utilizar Gradient Boosting como tarea de elección abierta debido a que ya hemos utizado este modelo en un evento externo a la universidad (Hackathon). El tiempo en el evento era muy limitado por lo que no pudimos aprender bien sobre este modelo y nos quedamos con ganadas de poder implementarlo de mejor forma.

In [ ]:
import time
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

#Silenciar los logs de proceso de cada trial de Optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

pipeline_gb = Pipeline(steps=[
    ('preprocesamiento', preprocessor_opt),
    ('clasificador', GradientBoostingClassifier(random_state=SEED))
])

start_time = time.time()

scores_gb_base = cross_val_score(
    pipeline_gb,
    X_train_full,
    y_train_full,
    cv=inner_cv,
    scoring='roc_auc',
    n_jobs=-1
)

tiempo_gb_base = time.time() - start_time
roc_auc_gb_base = scores_gb_base.mean()

print("Gradient Boosting base")
print(f"ROC-AUC medio en CV: {roc_auc_gb_base:.4f}")
print(f"Tiempo total: {tiempo_gb_base:.2f} segundos")

def objective_gb(trial):
    #Espacio de búsqueda de hiperparámetros
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 1, 5),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'max_features': trial.suggest_categorical('max_features', [None, 'sqrt', 'log2'])
    }

    #Construimos pipeline con esos hiperparámetros
    model = Pipeline(steps=[
        ('preprocesamiento', preprocessor_opt),
        ('clasificador', GradientBoostingClassifier(
            random_state=SEED,
            **params
        ))
    ])

    #Evaluamos con validación cruzada inner
    scores = cross_val_score(
        model,
        X_train_full,
        y_train_full,
        cv=inner_cv,
        scoring='roc_auc',
        n_jobs=-1
    )

    #Optuna maximizará esta métrica
    return scores.mean()

start_time = time.time()

study_gb = optuna.create_study(
    direction='maximize',
    study_name='gradient_boosting_optuna'
)

study_gb.optimize(
    objective_gb,
    n_trials=40,
    show_progress_bar=False
)

tiempo_gb_hpo = time.time() - start_time

print(f"Mejor ROC-AUC en CV: {study_gb.best_value:.4f}")
print("Mejores hiperparámetros encontrados:")
print(study_gb.best_params)
print(f"Tiempo total de HPO: {tiempo_gb_hpo:.2f} segundos")

#Entrenamos con todo el train
best_gb = Pipeline(steps=[
    ('preprocesamiento', preprocessor_opt),
    ('clasificador', GradientBoostingClassifier(
        random_state=SEED,
        **study_gb.best_params
    ))
])

best_gb.fit(X_train_full, y_train_full)


#Comparamos el modelo base frente al modelo optimizado.
df_gb_resumen = pd.DataFrame({
    'Modelo': ['Gradient Boosting (base)', 'Gradient Boosting (Optuna)'],
    'ROC-AUC inner': [roc_auc_gb_base, study_gb.best_value],
    'Tiempo (s)': [tiempo_gb_base, tiempo_gb_hpo]
})

print("\nResumen de resultados de Gradient Boosting")
display(df_gb_resumen)

optuna.visualization.matplotlib.plot_optimization_history(study_gb)
plt.title("Historial de optimización - Gradient Boosting")
plt.tight_layout()
plt.show()

fig = optuna.visualization.matplotlib.plot_param_importances(study_gb)
plt.tight_layout()
plt.show()

feature_names_gb = best_gb.named_steps['preprocesamiento'].get_feature_names_out()
importancias_gb = best_gb.named_steps['clasificador'].feature_importances_

df_importancias_gb = pd.DataFrame({
    'variable': feature_names_gb,
    'importancia': importancias_gb
}).sort_values('importancia', ascending=False)

print("\nVariables más importantes según Gradient Boosting")
display(df_importancias_gb.head(15))

top_n = 15
top_features = df_importancias_gb.head(top_n).iloc[::-1]  #invertimos para que la más importante quede arriba

plt.figure(figsize=(10, 6))
plt.barh(top_features['variable'], top_features['importancia'])
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.title(f"variables más importantes - Gradient Boosting")
plt.tight_layout()
plt.show()

### Conclusiones de la tarea abierta

El modelo de Gradient Boosting tiene un rendimiento sólido en la evaluación mediante validación cruzada inner, alcanzando un ROC-AUC de 0.9209 con hiperparámetros por omisión.

Tras la optimización de hiperparametros mediante Optuna, el rendimiento mejora hasta un ROC-AUC de 0.9296, lo que indica que el ajuste de hiperparámetros ayuda a mejorar la preducción del modelo.

Sin embargo, esta mejora supone un coste compuacional mayor pasando de aproximadamente 5 segundos en el modelo base a 280 segundos durante el proceso de optimización.

En cuanto a los hiperparámetros óptimos, se observa que:
- Se utilizan un número elevado de estimadores (n_estimators = 397), lo que indica la necesidad de un modelo más complejo.
- La profundidad máxima de los árboles (max_depth = 5) sugiere que el modelo captura relaciones no lineales moderadamente complejas.

En conjunto, estos resultados confirman que Gradient Boosting es una buena alternativa para este problema, que peude mejorar ligeramente el rendimiento respecto a modelos anteriores, aunque a costa de un mayor tiempo de entrenamiento.